In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy  as np

In [2]:

from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CERerankingEvaluator



In [3]:

# from sentence_transformers.cross_encoder.evaluation import CERerankingEvaluator

# 1. Load the small base model
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1)

# Load your data
df = pd.read_csv("final_dataframe.csv")

# Assuming your dataframe has columns: 'resume', 'job_description', 'ats_score'
X = df[['resume', 'jd']]
y = df['new_ats']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [4]:
df.head()

,resume,jd,new_ats
0,Abhishek Chaudhary \n ac5712916@gmail.com |...,Full job description\nPosition: Junior PHP Dev...,75
1,Abhishek Chaudhary \n ac5712916@gmail.com |...,Full job description\nJob Summary:\n\nWe are s...,65
2,Abhishek Chaudhary \n ac5712916@gmail.com |...,Job Title: Sr. Website UI/UX Designer and Deve...,65
3,Abhishek Chaudhary \n ac5712916@gmail.com |...,We are looking for a passionate and enthusiast...,85
4,Abhishek Chaudhary \n ac5712916@gmail.com |...,Full job description\nJob Title: Software Engi...,65


In [5]:


# 2. Prepare Data into InputExamples
train_samples = []
for index, row in X_train.iterrows():
    score = y_train.loc[index]
    # Ensure the score is a float
    train_samples.append(InputExample(texts=[row['resume'], row['jd']], label=float(score)))

# The DataLoader wraps our training samples
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=16)

# 3. Fine-Tune the model
model.fit(
    train_dataloader=train_dataloader,
    epochs=1,
    warmup_steps=100,
    output_path="Testing//DividedDataset//model"
)
# 4. Prepare evaluation data
test_samples = []
for index, row in X_test.iterrows():
    score = y_test.loc[index]
    test_samples.append(InputExample(texts=[row['resume'], row['jd']], label=float(score)))


model.save("DividedDataset//model-final")




Token indices sequence length is longer than the specified maximum sequence length for this model (643 > 512). Running this sequence through the model will result in indexing errors
D:\DS Projects\Testing\ats_env\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


KeyError: 'job_description'

In [12]:

model_path = "DividedDataset//model-final" 
print(f"Loading model from: {model_path}")
model = CrossEncoder(model_path)


# =====================================================================================
# Step 2: Prepare the test data and make predictions
# =====================================================================================

# The model.predict method expects a list of text pairs.
# We create this list from your X_test DataFrame.
print("Preparing test data for prediction...")
test_texts = [list(row) for index, row in X_test.iterrows()]

print(f"Making predictions on {len(test_texts)} test samples...")
# Use the model to predict scores on the test set
# show_progress_bar=True is helpful for large test sets
predicted_scores = model.predict(test_texts, show_progress_bar=True)


# =====================================================================================
# Step 3: Evaluate the model's performance
# =====================================================================================

# Get the actual scores from your test set
actual_scores = y_test.values

# --- Quantitative Evaluation: Calculate Metrics ---
print("\n--- Quantitative Evaluation ---")

# Mean Squared Error (MSE): Measures the average of the squares of the errors. Lower is better.
mse = mean_squared_error(actual_scores, predicted_scores)
print(f"Mean Squared Error (MSE): {mse:.4f}")

# Mean Absolute Error (MAE): Measures the average magnitude of the errors. It's in the same unit as the score. Lower is better.
mae = mean_absolute_error(actual_scores, predicted_scores)
print(f"Mean Absolute Error (MAE): {mae:.4f}")

# Root Mean Squared Error (RMSE): The square root of MSE. Easier to interpret as it's in the same unit as the score.
rmse = np.sqrt(mse)
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")


# --- Qualitative Evaluation: Inspect Some Predictions ---
print("\n--- Qualitative Evaluation (Sample Predictions) ---")

# Create a DataFrame to easily compare actual vs. predicted scores
results_df = X_test.copy()
results_df['actual_score'] = actual_scores
results_df['predicted_score'] = predicted_scores
results_df['error'] = abs(results_df['actual_score'] - results_df['predicted_score'])

# Display the 5 predictions with the smallest error (best predictions)
print("\nTop 5 Best Predictions (Lowest Error):")
print(results_df.nsmallest(5, 'error'))

# Display the 5 predictions with the largest error (worst predictions)
print("\nTop 5 Worst Predictions (Highest Error):")
print(results_df.nlargest(5, 'error'))

Loading model from: DividedDataset//model-final
Preparing test data for prediction...
Making predictions on 158 test samples...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


--- Quantitative Evaluation ---
Mean Squared Error (MSE): 3366.6970
Mean Absolute Error (MAE): 55.8144
Root Mean Squared Error (RMSE): 58.0232

--- Qualitative Evaluation (Sample Predictions) ---

Top 5 Best Predictions (Lowest Error):
                                                 resume  \
1190  Gaurav bhagat\nSoave Engineer\n2 Profile\nHigh...   
886   Vani Thakur \nStudent \nTo obtain a position w...   
251   RESUME \r\nRATAN MISHRA \r\n+91 95082 02947 \r...   
49    About: \nANAJNI BHARDWAJ \nLinkedin| contact n...   
59    About: \nANAJNI BHARDWAJ \nLinkedin| contact n...   

                                                     jd  actual_score  \
1190  We are looking for a passionate and enthusiast...            25   
886   Job Title: Sr. Website UI/UX Designer and Deve...            25   
251   Embark on a transformative journey as a Data A...            25   
49    Full job description\nJob Summary:\n\nWe are s...            25   
59     Job Summary:  The role plays a Plays